# Midsem Q5: TF-IDF Matrix, Class Vectors & Doc2Vec

**Given Documents:**

| Document | Text | Class |
|----------|------|-------|
| D1 | machine learning is useful | Technology |
| D2 | machine learning is powerful | Technology |
| D3 | learning language is useful | Language |
| D4 | language processing is useful | Language |

**Tasks:**
1. Construct vocabulary and compute TF-IDF of each word (no start/end markers).
2. Represent classes using TF-IDF vectors of documents.
3. How to learn document representations through Doc2Vec (name the technique and draw architecture).

## Part 1: Vocabulary & TF-IDF Matrix

In [ ]:
import numpy as np
import pandas as pd

# Documents
documents = {
    "D1": "machine learning is useful",
    "D2": "machine learning is powerful",
    "D3": "learning language is useful",
    "D4": "language processing is useful",
}
classes = {"D1": "Technology", "D2": "Technology", "D3": "Language", "D4": "Language"}

# Tokenize
doc_tokens = {doc_id: text.split() for doc_id, text in documents.items()}

# Build vocabulary (sorted alphabetically)
vocab = sorted(set(word for tokens in doc_tokens.values() for word in tokens))
N = len(documents)

print(f"Number of Documents (N): {N}")
print(f"Vocabulary ({len(vocab)} words): {vocab}")

In [ ]:
# ─────────────────────────────────────────────
# Term Frequency (TF)
# TF(t, d) = count(t in d) / |d|
# ─────────────────────────────────────────────

tf_matrix = {}
for doc_id, tokens in doc_tokens.items():
    doc_len = len(tokens)
    tf_matrix[doc_id] = {}
    for word in vocab:
        count = tokens.count(word)
        tf_matrix[doc_id][word] = count / doc_len

df_tf = pd.DataFrame(tf_matrix).T
df_tf = df_tf[vocab]  # reorder columns
print("TF (Term Frequency) Matrix:")
print("TF(t,d) = count(t in d) / |d|  where |d| = 4 for all documents")
print()
display(df_tf)

In [ ]:
# ─────────────────────────────────────────────
# Document Frequency (DF) & IDF
# IDF(t) = log(N / DF(t))   using natural log
# ─────────────────────────────────────────────

df_counts = {}
for word in vocab:
    df_counts[word] = sum(1 for tokens in doc_tokens.values() if word in tokens)

idf = {word: np.log(N / df_count) for word, df_count in df_counts.items()}

print("Document Frequency (DF) and IDF:")
print(f"{'Word':<15} {'DF':>3}  {'IDF = ln(N/DF)':>15}")
print("─" * 38)
for word in vocab:
    print(f"{word:<15} {df_counts[word]:>3}  {idf[word]:>15.4f}")

In [ ]:
# ─────────────────────────────────────────────
# TF-IDF Matrix = TF × IDF
# ─────────────────────────────────────────────

tfidf_matrix = {}
for doc_id in documents:
    tfidf_matrix[doc_id] = {}
    for word in vocab:
        tfidf_matrix[doc_id][word] = tf_matrix[doc_id][word] * idf[word]

df_tfidf = pd.DataFrame(tfidf_matrix).T
df_tfidf = df_tfidf[vocab]

print("\n" + "=" * 70)
print("TF-IDF MATRIX")
print("TF-IDF(t, d) = TF(t, d) × IDF(t) = TF(t, d) × ln(N / DF(t))")
print("=" * 70)
display(df_tfidf.round(4))

print("\nKey Observations:")
print("  • 'is' has TF-IDF = 0.0000 everywhere (appears in all 4 docs → IDF = ln(1) = 0)")
print("  • 'powerful' and 'processing' have the highest TF-IDF values")
print("    (each appears in only 1 document → IDF = ln(4) ≈ 1.3863)")
print("  • Words common across many documents get lower TF-IDF scores")

## Part 2: Class Representation via TF-IDF Vectors

To represent each **class** using TF-IDF, we compute the **mean (centroid) of the TF-IDF vectors** of all documents belonging to that class:

$$\vec{C}_{\text{class}} = \frac{1}{|\text{class}|} \sum_{d \in \text{class}} \text{TF-IDF}(d)$$

In [ ]:
# Class vectors = mean of member document TF-IDF vectors
class_docs = {"Technology": ["D1", "D2"], "Language": ["D3", "D4"]}

class_vectors = {}
for cls, doc_ids in class_docs.items():
    vectors = [list(tfidf_matrix[d].values()) for d in doc_ids]
    class_vectors[cls] = np.mean(vectors, axis=0)

df_class = pd.DataFrame(class_vectors, index=vocab).T

print("Class Vectors (Mean TF-IDF of member documents):")
print(f"  Technology = mean(D1, D2)")
print(f"  Language   = mean(D3, D4)")
print()
display(df_class.round(4))

print("\nInterpretation:")
print("  • Technology class has high weight on 'machine' (unique to this class)")
print("  • Language class has high weight on 'language' (characteristic of this class)")
print("  • To classify a new document, compute its TF-IDF vector and find")
print("    the nearest class centroid (e.g., using cosine similarity).")

## Part 3: Learning Document Representations via Doc2Vec

### Technique: **Paragraph Vector – Distributed Memory (PV-DM)**

Doc2Vec (Le & Mikolov, 2014) extends Word2Vec by introducing a **paragraph/document vector** that acts as a memory for the document context.

### PV-DM Architecture:

```
┌──────────────────────────────────────────────────────────┐
│                      PV-DM Architecture                  │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  INPUT LAYER:                                            │
│  ┌───────────┐  ┌──────┐  ┌──────┐  ┌──────┐            │
│  │Document ID │  │ w(t-2)│  │ w(t-1)│  │ w(t+1)│           │
│  │  (D_i)     │  │      │  │      │  │      │            │
│  └─────┬─────┘  └──┬───┘  └──┬───┘  └──┬───┘            │
│        │            │         │          │                │
│        ▼            ▼         ▼          ▼                │
│  ┌───────────────────────────────────────────┐           │
│  │  Paragraph Matrix D    Word Matrix W       │           │
│  │  (lookup embeddings)   (lookup embeddings) │           │
│  └─────────────────┬─────────────────────────┘           │
│                    │                                      │
│                    ▼                                      │
│          ┌─────────────────────┐                         │
│          │  Average / Concat    │                         │
│          │  (combine vectors)   │                         │
│          └──────────┬──────────┘                         │
│                     │                                     │
│                     ▼                                     │
│          ┌─────────────────────┐                         │
│          │   Hidden Layer       │                         │
│          │   (fully connected)  │                         │
│          └──────────┬──────────┘                         │
│                     │                                     │
│                     ▼                                     │
│          ┌─────────────────────┐                         │
│          │   Softmax Output     │                         │
│          │   Predict w(t)       │                         │
│          └─────────────────────┘                         │
│                                                          │
│  OBJECTIVE: Maximize P(w_t | D_i, w_{t-k}, ..., w_{t-1})│
└──────────────────────────────────────────────────────────┘
```

### Key Differences from Word2Vec:
- An additional **document/paragraph ID vector** is concatenated or averaged with the context word vectors.
- The document vector acts as a **memory** that remembers the topic of the document.
- During training, both the **document matrix D** and **word matrix W** are updated via backpropagation.
- At inference time (new document), word matrix W is frozen and only the new document vector is learned via gradient descent.

### Training:
- **Objective**: Predict the target word $w_t$ given the document ID and surrounding context words.
- **Loss**: Cross-entropy loss over the vocabulary.
- **Backpropagation**: Gradients flow back through the softmax → hidden layer → both document and word embeddings.